[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremylongshore/claude-code-plugins-plus-skills/blob/main/tutorials/skills/01-what-is-skill.ipynb)

# What is SKILL.md?

**Learning Path**: Skills → Plugins → Orchestration  
**Level**: Beginner  
**Time**: 20 minutes  
**Goal**: Understand what SKILL.md files are and why they exist

---

## The Problem: Repeated Context

**Without Skills**:
```
You: "Hey Claude, parse these BigQuery schemas and generate migrations.
     Oh, and use snake_case for field names, add timestamps..."
     
[Next conversation - copy-paste same instructions again]
```

**With Skills**:
```
You: "Generate schema migrations"

Claude: [Automatically uses bigquery-migrations skill]
        [Already knows your standards]
```

**The difference**: Skills are **persistent, reusable instructions**.

In [ ]:
# Example: What is a Skill?
skill_definition = {
    "what": "Filesystem-based capability package",
    "format": "SKILL.md file with YAML frontmatter + markdown body",
    "discovery": "Automatic - Claude finds and loads at startup",
    "invocation": "Intent-matched - Claude decides when to use it",
    "persistence": "Works across ALL conversations"
}

print("WHAT IS A CLAUDE SKILL?")
print("=" * 60)
for aspect, explanation in skill_definition.items():
    print(f"{aspect.upper()}: {explanation}")

print("\n💡 Mental Model: Like an onboarding guide for Claude!")

## Skills vs Ad-Hoc Prompts

| Aspect | Ad-Hoc Prompts | Skills |
|--------|---------------|--------|
| Reusability | One conversation | Persistent |
| Discovery | Manual | Automatic |
| Organization | Scattered | Structured |
| Context | Full load | Progressive |
| Code | Generated | Pre-written |
| Tool Auth | Ask each time | Pre-approved |

In [ ]:
# Compare context usage
print("AD-HOC APPROACH")
print("=" * 60)
print('User: "Parse schema, use snake_case, add timestamps,"')
print('      "validate names, check duplicates, generate migration..."')
print("Context: ~2000 tokens EVERY conversation")
print("")

print("SKILL APPROACH")
print("=" * 60)
print('User: "Generate schema migration"')
print("Claude: Loads bigquery-migrations skill")
print("Context: ~200 tokens (metadata only)")
print("")
print("💡 Skills = 10x less context, 100x more consistent")

## Where Skills Live

Skills are discovered in **4 locations** with priority:

| Location | Scope | Priority |
|----------|-------|----------|
| `~/.claude/skills/` | Personal | 1 (lowest) |
| `.claude/skills/` | Project | 2 |
| Plugin `skills/` | Marketplace | 3 |
| Built-in | Platform | 4 (highest) |

**Priority Rule**: Higher priority wins on name conflicts

In [ ]:
# Skill discovery locations
locations = [
    {"path": "~/.claude/skills/", "priority": 1, "use": "Personal tools"},
    {"path": ".claude/skills/", "priority": 2, "use": "Project workflows"},
    {"path": "plugins/.../skills/", "priority": 3, "use": "Marketplace skills"},
    {"path": "Built-in", "priority": 4, "use": "Core capabilities"}
]

print("SKILL DISCOVERY LOCATIONS")
print("=" * 60)
for loc in locations:
    print(f"\n📁 {loc['path']}")
    print(f"   Priority: {loc['priority']}")
    print(f"   Use for: {loc['use']}")

print("\n💡 Same name? Higher priority wins!")

## The Skill Tool Architecture

### Critical Insight: Skills Are NOT in System Prompt

Skills live in a **meta-tool** called `Skill`:

```javascript
tools: [
  { name: "Read", ... },
  { name: "Write", ... },
  {
    name: "Skill",
    inputSchema: { command: string },
    description: "<available_skills>..."
  }
]
```

### How Invocation Works

1. User types request
2. Claude analyzes intent
3. Matches to skill description
4. Calls `Skill` tool with skill name
5. Skill body loads into context
6. Claude follows skill instructions

In [ ]:
# Simulate skill invocation
def invoke_skill(user_request):
    print(f"User: '{user_request}'\n")
    print("Step 1: Claude checks available skills")
    print("  - bigquery-migrations: 'Generate schema migrations'")
    print("  - security-scanner: 'Scan for vulnerabilities'")
    print("  - commit-helper: 'Generate git commits'")
    print("")
    
    if "schema" in user_request.lower():
        skill = "bigquery-migrations"
        print(f"Step 2: Matched -> '{skill}'")
        print(f"Step 3: Call Skill('{skill}')")
        print("Step 4: Load SKILL.md into context")
        print("Step 5: Execute skill instructions")
    else:
        print("Step 2: No skill match - use general capabilities")

invoke_skill("Generate a migration for users table")

## 🎯 Try It: Parse a Real SKILL.md

In [ ]:
# Example SKILL.md (Intent Solutions standard)
example_skill = """---
name: bigquery-migrations
description: |
  Generate BigQuery schema migrations with validation.
  Use when: migrating schemas, adding fields, changing types.
allowed-tools: Read,Write,Grep,Bash(bq:*)
version: 1.0.0
author: Jane Developer <jane@example.com>
license: MIT
---

# BigQuery Migration Generator

Generate safe, validated BigQuery schema migrations from your current
schema state to a desired target state.

## Overview

This skill reads your existing BigQuery schemas, compares them against
requested changes, and produces validated DDL migration scripts.

## Prerequisites

- `bq` CLI installed and authenticated
- Target dataset must exist
- IAM permissions for schema reads

## Instructions

1. Read current schema with `bq show`
2. Analyze requested changes
3. Generate migration DDL
4. Validate column types and constraints
5. Output migration file

For detailed implementation patterns, see:
`${CLAUDE_SKILL_DIR}/references/implementation.md`

## Output

- Migration `.sql` file in `migrations/` directory
- Validation summary printed to console

## Error Handling

- If dataset not found, prompt user for correct project/dataset
- If type conflict detected, abort and list conflicts

## Examples

```
User: "Add a created_at timestamp to users table"
→ Generates: migrations/002_add_created_at.sql
```

## Resources

- [BigQuery DDL reference](https://cloud.google.com/bigquery/docs/reference/standard-sql/data-definition-language)
- See `${CLAUDE_SKILL_DIR}/references/implementation.md` for advanced patterns
"""

# Parse it
parts = example_skill.split('---')
frontmatter = parts[1].strip()
body = parts[2].strip()

print("PARSED SKILL.MD")
print("=" * 60)
print("\nFRONTMATTER (YAML) — 6 required fields:")
print(frontmatter)
print("\nBODY (Markdown) — 7 required sections:")
sections = [line for line in body.split('\n') if line.startswith('## ')]
for s in sections:
    print(f"  ✅ {s}")
print(f"\nBody length: {len(body.splitlines())} lines")
print("Target: ≤150 lines (500 max). Heavy content → references/")
print("\n✅ This follows the Intent Solutions standard!")

## Key Takeaways

### What You Learned

1. **What is SKILL.md**: Filesystem capability package
2. **Why use skills**: Persistent vs one-time prompts
3. **Where skills live**: 4 locations with priority
4. **How they work**: Skill meta-tool + intent matching
5. **Standards matter**: Intent Solutions standard requirements

### Skill Standard Checklist

**6 Required Frontmatter Fields:**
- [ ] `name` — kebab-case
- [ ] `description` — includes "Use when..." and trigger phrases
- [ ] `allowed-tools` — CSV string, Bash always scoped (e.g., `Bash(git:*)`, `Bash(npm:*)`)
- [ ] `version` — SemVer
- [ ] `author` — Name and email
- [ ] `license` — e.g., MIT

**Optional Frontmatter Fields:**
`model` (`sonnet`, `haiku`, `opus`), `context`, `agent`, `user-invocable`, `argument-hint`, `hooks`, `compatibility`, `compatible-with`, `tags`

**7 Required Body Sections:**
- [ ] Overview
- [ ] Prerequisites
- [ ] Instructions
- [ ] Output
- [ ] Error Handling
- [ ] Examples
- [ ] Resources

**Size & Structure:**
- [ ] Purpose statement: 1-2 sentences, ≤400 chars (first paragraph after `# Title` or in `## Overview`)
- [ ] Body target: ≤150 lines (500 max)
- [ ] Heavy content offloaded to `references/` via Progressive Disclosure Architecture (PDA)
- [ ] Path references use `${CLAUDE_SKILL_DIR}` (e.g., `${CLAUDE_SKILL_DIR}/references/implementation.md`)

---

## Next Steps

1. **[02-skill-anatomy.ipynb](02-skill-anatomy.ipynb)** - Deep dive
2. **[03-build-your-first-skill.ipynb](03-build-your-first-skill.ipynb)** - Hands-on
3. **[04-advanced-patterns.ipynb](04-advanced-patterns.ipynb)** - Multi-phase

---

*Intent Solutions Standard Compliant - Version 1.0.0 - MIT License*